# Part 2a: Quantization with QKeras

In this notebook we retrain the jet tagger from Part 1 using **QKeras** (Quantized Keras). With quantization-aware training (QAT), the model is trained with low-precision, fixed-point weights, so the optimizer can correct the effect of quantization during training rather than after — enabling lower precisions (and thus resource consumption) without sacrificing accuracy.

Make sure you have run `1_getting_started/1a_train_keras.ipynb` first, as we load its saved data and use its model as a baseline for comparison.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
import plotting

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

## Load the jet tagging dataset

We load the preprocessed arrays saved by `1a_train_keras.ipynb`.

In [ ]:
X_train_val = np.load('../data/jet-tagging/X_train_val.npy')
X_test      = np.load('../data/jet-tagging/X_test.npy')
y_train_val = np.load('../data/jet-tagging/y_train_val.npy')
y_test      = np.load('../data/jet-tagging/y_test.npy')
classes     = np.load('../data/jet-tagging/classes.npy', allow_pickle=True)

## Construct a model

This time we're going to use QKeras layers.

**Note:** QKeras (https://github.com/google/qkeras) was originally developed for Keras v2. While the official version has not be an updated to Keras v3, the hls4ml community maintains a fork compatible with Keras v3: https://github.com/fastmachinelearning/qkerasV3. In this tutorial, we use the v3 version; though hls4ml still supports the v2 version of QKeras, with exactly the same syntax.


In [ ]:
from keras.models import Sequential
from keras.optimizers import Adam
from keras.layers import Activation
from qkeras.qlayers import QDense, QActivation
from qkeras.quantizers import quantized_bits, quantized_relu

We use `QDense` instead of `Dense`, and `QActivation` instead of `Activation`.
We specify `kernel_quantizer = quantized_bits(6, 0, alpha=1)`, which uses 6 bits (of which 0 are integer bits) for the weights and biases, and `quantized_relu(6)` for 6-bit ReLU activations.

In [ ]:
model = Sequential()
model.add(
    QDense(
        64,
        input_shape=(16,),
        name='fc1',
        kernel_quantizer=quantized_bits(6, 0, alpha=1),
        bias_quantizer=quantized_bits(6, 0, alpha=1),
    )
)
model.add(QActivation(activation=quantized_relu(6), name='relu1'))
model.add(
    QDense(
        32,
        name='fc2',
        kernel_quantizer=quantized_bits(6, 0, alpha=1),
        bias_quantizer=quantized_bits(6, 0, alpha=1),
    )
)
model.add(QActivation(activation=quantized_relu(6), name='relu2'))
model.add(
    QDense(
        32,
        name='fc3',
        kernel_quantizer=quantized_bits(6, 0, alpha=1),
        bias_quantizer=quantized_bits(6, 0, alpha=1),
    )
)
model.add(QActivation(activation=quantized_relu(6), name='relu3'))
model.add(
    QDense(
        5,
        name='output',
        kernel_quantizer=quantized_bits(6, 0, alpha=1),
        bias_quantizer=quantized_bits(6, 0, alpha=1),
    )
)
model.add(Activation(activation='softmax', name='softmax'))
model.summary()

## Train the model

We use the Adam optimiser with categorical crossentropy loss.
The model isn't very complex, so this should take just a few minutes even on a CPU.

In [ ]:
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(
    X_train_val,
    y_train_val,
    batch_size=1024,
    epochs=30,
    validation_split=0.25,
    shuffle=True,
)
os.makedirs('../models', exist_ok=True)
model.save('../models/qkeras_model_part2.h5')

## Check performance

How does the quantized model compare against the baseline from Part 1? Let's report the accuracy and make a ROC curve.
The baseline is shown with solid lines, the quantized model with dashed lines.

We should also check that hls4ml can respect the choice to use 6-bits throughout the model and match the accuracy. We'll generate a configuration from this quantized model and plot its performance as the dotted line.
The generated configuration is printed out. You'll notice that it uses 7 bits for the type, but we specified 6 — that's because QKeras doesn't count the sign bit, so the type that actually gets used needs 1 more.

We also use the `OutputRoundingSaturationMode` optimizer pass of hls4ml to set the Activation layers to round rather than truncate the cast. This is important for getting good model accuracy at small bit precision. 

In [ ]:
import hls4ml

config = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')
config['LayerName']['softmax']['exp_table_t'] = 'ap_fixed<18,8>'
config['LayerName']['softmax']['inv_table_t'] = 'ap_fixed<18,4>'
print('-----------------------------------')
plotting.print_dict(config)
print('-----------------------------------')
hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    backend='Vitis',
    output_dir='../hls4ml_prjs/hls4ml_prj_qkeras_part2',
    part='xcu250-figd2104-2L-e',
)
hls_model.compile()

y_qkeras = model.predict(np.ascontiguousarray(X_test))
y_hls    = hls_model.predict(np.ascontiguousarray(X_test))

In [ ]:
from sklearn.metrics import accuracy_score
from keras.models import load_model

model_ref = load_model('../models/keras_model_part1.h5')
y_ref = model_ref.predict(np.ascontiguousarray(X_test))

print('Accuracy baseline:  {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_ref,    axis=1))))
print('Accuracy quantized: {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_qkeras, axis=1))))
print('Accuracy hls4ml:    {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls,    axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_ref, list(classes))
plt.gca().set_prop_cycle(None)
_ = plotting.makeRoc(y_test, y_qkeras, list(classes), linestyle='--')
plt.gca().set_prop_cycle(None)
_ = plotting.makeRoc(y_test, y_hls, list(classes), linestyle=':')

from matplotlib.lines import Line2D
from matplotlib.legend import Legend
lines = [Line2D([0], [0], ls='-'), Line2D([0], [0], ls='--'), Line2D([0], [0], ls=':')]
leg = Legend(ax, lines, labels=['baseline', 'quantized', 'hls4ml'], loc='lower right', frameon=False)
ax.add_artist(leg)

## Synthesize

Now let's synthesize this quantized model.

**This can take several minutes.**


In [ ]:
hls_model.build(csim=False)

In [ ]:
hls4ml.report.read_vivado_report('../hls4ml_prjs/hls4ml_prj_quantized_part2')

Compare the DSP count above against the Part 1c baseline. With 6-bit quantization, every multiplication in this network is narrower than the ~10-bit threshold below which Vivado maps multiplications to LUT logic rather than DSP slices, significantly reducing the DSP usage.

## A note on HLS synthesis resource estimates


The resource numbers reported by Vitis HLS after HLS C-synthesis are **estimates** derived from the HLS's internal model and often do not truly reflect the final resource consumption of the model. These estimates **often overestimate** LUT consumption, sometimes by an order of magnitude.

For a more accurate picture of resource consumption, you should run **Vivado synthesis** (`vsynth`). This invokes the full Vivado synthesis flow on the generated RTL, producing estimates that are much closer to what you would see after implementation (place-and-route).

**This step can take 10–20 minutes.**

In [ ]:
hls_model.build(reset=False, csim=False, vsynth=True)

In [ ]:
hls4ml.report.read_vivado_report('../hls4ml_prjs/hls4ml_prj_quantized_part2')